In [16]:
from typing import List, TypedDict, Union
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from litellm.llms.vllm.completion.handler import llm
import os

from pygments.lexers import find_lexer_class_by_name

In [17]:
load_dotenv()

API_KEY = os.getenv("API_KEY")
MODEL = os.getenv("MODEL")
BASE_URL = os.getenv("BASE_URL")


In [18]:
class AgentState(TypedDict):
    message: List[Union[HumanMessage, AIMessage]]

llm = ChatOpenAI(model=MODEL, api_key=API_KEY, base_url=BASE_URL)

def process(state: AgentState) -> AgentState:
    """ This node will solve the request you input """

    response = llm.invoke(state["message"])
    state["message"].append(AIMessage(content=response.content))
    print(f"\nAI: {response.content}")
    return state

In [19]:
graph = StateGraph(AgentState)

PROCESS = "process"

graph.add_node(PROCESS, process)
graph.add_edge(START, PROCESS)
graph.add_edge(PROCESS, END)

agent = graph.compile()

In [20]:
conversation_history = []

In [21]:
user_input = input("Enter: ")
while user_input != "exit":
    conversation_history.append(HumanMessage(content=user_input))

    result = agent.invoke({"message": conversation_history})
    conversation_history = result["message"]

    user_input = input("Enter : ")


AI: You can call me Nemotron, and I'm a language model trained by NVIDIA. I can help with a wide variety of tasks—answering questions, brainstorming ideas, writing, coding, learning new topics, and more. 

What's on your mind, or how can I help you today?

AI: Here's a classic one for you:

🔬 **Why don't scientists trust atoms?**  
*Because they make up everything!*

😄 Let me know if you'd like another!

AI: I did! 😄 

Did you like it? Want to hear another one, or should I switch to a different topic? I'm full of these!

AI: 🌾 **Why did the scarecrow win an award?**  
*Because he was outstanding in his field!*

Heh. Want another, or are you officially a "pun master" now? 😄

AI: No problem! Whenever you're ready, I'm here if you want another joke, help with something, or just want to chat. Have a great one! 👋


In [22]:
file_name = "log.txt"

with open(file_name, "w") as file:
    file.write("Conversation log:\n")

    for message in conversation_history:
        if isinstance(message, HumanMessage):
            file.write(f"You: {message.content}\n")
        if isinstance(message, AIMessage):
            file.write(f"AI: {message.content}\n")
    file.write("End of conversation")

print(f"Conversation saved to {file_name}\n")

Conversation saved to log.txt

